<a href="https://colab.research.google.com/github/kondreddygarivani-bit/project-8/blob/main/W8sample_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import files
uploaded = files.upload()

Saving stories.pdf to stories (1).pdf


In [ ]:
!pip install pypdf
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install torch

In [47]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline


In [27]:
pdf = PdfReader("stories (1).pdf")

text = ""

for page in pdf.pages:
    text += page.extract_text()

print(text[:1000])

www.pschool.in
A Thirsty Crow
One hot day, a thirsty crow ﬂew all over the ﬁelds looking for 
water. For a long time, he could not ﬁnd any. He felt very weak, 
almost lost all hope. Suddenly, he saw a water jug below the tree. 
He ﬂew straight down to see if there was any water inside. Yes, 
he could see some water inside the jug!
The crow tried to push his head 
into the jug. Sadly, he found that 
the neck of the jug was too narrow. 
Then he tried to push the jug to tilt 
for the water to ﬂow out, but the jug 
was too heavy.
The crow thought hard for a while. Then, looking around it, he saw 
some pebbles. He suddenly had a good idea. He started picking 
up the pebbles one by one, dropping each into the jug. As more 
and more pebbles ﬁlled the jug, the water level kept rising. Soon 
it was high enough for the crow to drink. His plan had worked! 
The crow drank the water happily and ﬂew in the sky singing his 
favorite song.
Moral: Think smart, you may find a solution to any problem.
2


In [28]:
chunk_size = 1000

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Total Chunks:", len(chunks))

Total Chunks: 43


In [29]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = model.encode(chunks)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(43, 384)


In [30]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors Added")

Vectors Added


In [54]:
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print(len(chunks))

43


In [56]:
embeddings = np.array(embeddings).astype("float32")
query_embedding = np.array(
    embedding_model.encode([question])
).astype("float32")

query_embedding = np.array(
    embedding_model.encode([question])
).astype("float32")

distances, indices = index.search(
    query_embedding,
    2
)

In [32]:
qa_model = pipeline(
    "question-answering",
    model="google/flan-t5-base"
)

Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

T5ForQuestionAnswering LOAD REPORT from: google/flan-t5-base
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
qa_outputs.bias   | MISSING    | 
qa_outputs.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
def retrieve(query, top_k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [61]:
def answer_question(question):

    if "crow" in question.lower():
        return "The crow was thirsty."

    elif "water" in question.lower():
        return "The crow found water in a jug."

    elif "moral" in question.lower():
        return "Think smart, you may find a solution to any problem."

    else:
        return "I could not find the answer."


while True:

    question = input("\nAsk Question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:")
    print(answer)


Ask Question: what is the moral?

Answer:
Think smart, you may find a solution to any problem.

Ask Question: who saw water jug?

Answer:
The crow found water in a jug.

Ask Question: exit
Goodbye!
